# Setting SQL Safety Constraints

<figure>
 <img src="../assets/chapter_3_03.png" width="70%" align="center"/></a>
<figcaption> SQL Validation </figcaption>
</figure>

<br>
<br />

Common ones include:
- `LIMIT` / `TOP` / `FETCH FIRST` - Caps the number of rows returned to prevent large result sets.
- `WHERE` filters - Constrain which rows are scanned or returned (e.g., date ranges, partitions, tenant IDs).
- `ORDER BY` (often paired with `LIMIT`) - Ensures deterministic results, especially when returning only a subset of rows.
- Time or range constraints - For example, requiring a date predicate (`WHERE` event_date >= …) to avoid unbounded scans.
- Column constraints - Restricting queries to a subset of allowed columns (e.g., excluding PII or large blobs).
- Join constraints - Limiting the number or type of joins (e.g., disallowing cross joins or many-to-many joins).
- Execution constraints (engine-level) - Such as query timeouts, memory limits, or maximum bytes scanned.
- Result-shape constraints - Enforcing aggregations (GROUP BY) instead of raw row-level access for sensitive tables.

## Adding LIMIT constraint

The following function using the `sqlglot` parsing functionality to evaluate if a SQL query contains a LIMIT constraint, and if not set one. In addition, it enables users to define the limit size.

In [4]:
from sqlglot import parse_one, exp


def ensure_limit(
    sql: str,
    *,
    limit: int = 10,
    dialect: str | None = None,
) -> str:
    """
    Ensure a SQL query has a LIMIT clause.
    If missing, add LIMIT <limit>.

    Parameters
    ----------
    sql : str
        Input SQL query
    limit : int, default=10
        LIMIT value to enforce
    dialect : str, optional
        SQL dialect (e.g. "duckdb", "postgres", "snowflake")

    Returns
    -------
    str
        SQL query with a LIMIT clause
    """
    try:
        expression = parse_one(sql, dialect=dialect)
    except Exception:
        raise ValueError("Invalid SQL query")

    # Handle SELECT and WITH statements
    select = None

    if isinstance(expression, exp.Select):
        select = expression
    elif isinstance(expression, exp.With):
        select = expression.this

    if select is None:
        # Non-select statements are returned unchanged
        return sql

    # If LIMIT already exists, return as-is
    if select.args.get("limit") is not None:
        return expression.sql(dialect=dialect)

    # Add LIMIT
    select.set(
        "limit",
        exp.Limit(
            expression=exp.Literal.number(limit)
        ),
    )

    return expression.sql(dialect=dialect)

In [5]:
ensure_limit(sql="SELECT * FROM air_traffic", limit=10)

'SELECT * FROM air_traffic LIMIT 10'